#### data_check結果
・データ内容：人材<br>
・行動種別：全てあり<br>
　　1    1981780<br>
　　0    1286738<br>
　　3      74561<br>
　　2      33414<br>
・データ欠損：なし<br>
・重複行：17行あり

In [1]:
import pandas as pd

In [2]:
#ファイル読み込み
df = pd.read_csv('../000.data/train/train_A.tsv', sep="\t")

In [3]:
df_drop = df.drop('time_stamp', axis=1)

In [4]:
#各列の情報確認
df_drop.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3376493 entries, 0 to 3376492
Data columns (total 4 columns):
 #   Column      Dtype 
---  ------      ----- 
 0   user_id     object
 1   product_id  object
 2   event_type  int64 
 3   ad          int64 
dtypes: int64(2), object(2)
memory usage: 103.0+ MB


In [5]:
#行動種別(event_type)毎の数
df_drop['event_type'].value_counts()

event_type
1    1981780
0    1286738
3      74561
2      33414
Name: count, dtype: int64

In [6]:
#コンバージョンの数
df_drop['ad'].value_counts()

ad
-1    3301932
 1      57398
 0      17163
Name: count, dtype: int64

In [7]:
#データフレームのコピー
train = df_drop.copy()

In [8]:
#ユーザー毎の人気度を追加
train['user_interest'] = train[train['event_type'].isin([0, 1, 3])].groupby('user_id')['event_type'].transform('count')

In [9]:
#商品毎の人気度を追加
train['product_interest'] = train[train['event_type'].isin([0, 1, 3])].groupby('product_id')['event_type'].transform('count')

In [10]:
#user_idとproduct_idを数値化
train['user_id'] = train['user_id'].astype('category').cat.codes
train['product_id'] = train['product_id'].astype('category').cat.codes

In [11]:
# event_typeごとの割合
event_type_counts = train['event_type'].value_counts()
print("Original class distribution:\n", event_type_counts)

Original class distribution:
 event_type
1    1981780
0    1286738
3      74561
2      33414
Name: count, dtype: int64


In [12]:
# 総数を希望のサンプル数に設定
total_samples = 50000

In [13]:
# 各クラスの割合を保ちながら、サンプリング
class_proportions = event_type_counts / event_type_counts.sum()
sample_sizes = (class_proportions * total_samples).round().astype(int)

In [14]:
# サンプリング実施
sampled_data = pd.DataFrame()

for event_type, sample_size in sample_sizes.items():
    sampled_data = pd.concat([sampled_data, train[train['event_type'] == event_type].sample(sample_size, random_state=42)])

In [15]:
# サンプル後のデータ確認
sampled_event_type_counts = sampled_data['event_type'].value_counts()
print("Resampled class distribution:\n", sampled_event_type_counts)

Resampled class distribution:
 event_type
1    29347
0    19054
3     1104
2      495
Name: count, dtype: int64


In [16]:
print(sampled_data)

         user_id  product_id  event_type  ad  user_interest  product_interest
1613588    28144        6042           1  -1           78.0             611.0
150315     11386        1923           1  -1          756.0             289.0
1710353    32887        3301           1  -1          458.0             143.0
1080004    47708        9634           1  -1           48.0            2917.0
3099928    11558        8817           1  -1           86.0             910.0
...          ...         ...         ...  ..            ...               ...
2455844    46167        2833           2  -1            NaN               NaN
3036566    26853       13508           2  -1            NaN               NaN
3190306    13995       12009           2  -1            NaN               NaN
2768482    31289        1795           2  -1            NaN               NaN
2871957    30227        7886           2  -1            NaN               NaN

[50000 rows x 6 columns]


In [17]:
sampled_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 50000 entries, 1613588 to 2871957
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   user_id           50000 non-null  int32  
 1   product_id        50000 non-null  int16  
 2   event_type        50000 non-null  int64  
 3   ad                50000 non-null  int64  
 4   user_interest     49505 non-null  float64
 5   product_interest  49505 non-null  float64
dtypes: float64(2), int16(1), int32(1), int64(2)
memory usage: 2.2 MB


In [18]:
#前処理結果をcsvへ
sampled_data.to_csv('./train/train_A.csv', index_label=False)